# 📡 FreeAlphaRadar — Colab / Kaggle launcher

Zero-config: **no API keys, no sign-ups, no secrets**. This notebook clones the repo, installs the pinned dependencies, seeds the offline sample cache, and launches the Streamlit dashboard via a public tunnel.

If the runtime has no outbound network for the financial data sources, the app automatically serves the seeded sample data so everything still works.

## 1. Clone the repository

In [ ]:
import os
REPO_URL = "https://github.com/chizkidd/freealpharadar.git"
if not os.path.exists("freealpharadar"):
    !git clone $REPO_URL
%cd freealpharadar

## 2. Install dependencies (pinned for reproducibility)

In [ ]:
# Slim runtime + test deps (pytest is used by the sanity-check cell below).
!pip install -q -r requirements.txt -r requirements-dev.txt

# Optional: enable the real pre-trained FinBERT + XGBoost (~3.9 GB). Colab
# often has torch preinstalled. Skip this line to use the fast lexicon
# sentiment backend instead — the app works identically either way.
!pip install -q -r requirements-ml.txt

## 3. Seed the offline sample cache

This guarantees the dashboard is populated even with no network access.

In [ ]:
!python run_scorer.py --seed-sample --no-refresh --no-ml

## 4. Run the offline test-suite (optional sanity check)

In [ ]:
!FAR_OFFLINE=1 python -m pytest -q

## 5. Launch Streamlit through a public tunnel

We start Streamlit in the background and expose it with `localtunnel`. The cell prints a public URL and the tunnel password (which is just the Colab VM's external IP).

In [ ]:
# Install localtunnel (no account required).
!npm install -g localtunnel >/dev/null 2>&1

import subprocess, time
# Start Streamlit in the background (headless, no usage stats).
proc = subprocess.Popen(
    ["streamlit", "run", "streamlit_app.py",
     "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(8)
print("Tunnel password (your external IP):")
!curl -s https://loca.lt/mytunnelpassword || curl -s ifconfig.me
print("\nOpen the URL printed below and enter the password above:")
!npx --yes localtunnel --port 8501

### Alternative: `streamlit-jupyter` inline

If you prefer to render inside the notebook, install `streamlit-jupyter` and run the app object directly. The tunnel approach above is the most reliable on Colab/Kaggle.